# Set Up

In [1]:
#Import variables
import os
exec(open("./PGS_calc_param.txt").read())

#Import packages
from datetime import datetime
import psutil
import time
import os
import subprocess

#Get CPU number
logical_cpus = psutil.cpu_count(logical=True)
print(logical_cpus)

64


# Write Script

In [2]:
%%writefile /home/jupyter/workspace/workspace-bucket/calculate_pgs/plink_bed_parallel.sh
#!/bin/bash

set -o pipefail
set -o errexit

echo "=============================="
echo "PLINK JOB START"
echo "PGS=${PGS}"
echo "BUILD=${BUILD}"
echo "CHROMS=${CHROMS}"
echo "START=$(date)"
echo "=============================="

SECONDS=0

INPUT_PATH="/home/jupyter/workspace/vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed"
OUTPUT_BASE="/home/jupyter/workspace/workspace-bucket/calculate_pgs/plink_results"
WEIGHTS_PATH="/home/jupyter/workspace/workspace-bucket/calculate_pgs/pgs_bim_matched_weights"

MAX_JOBS="${MAX_JOBS:-1}"

RUN_ID="$(date +%Y%m%d_%H%M%S)"
OUTPUT_PATH="${OUTPUT_BASE}/${PGS}_${BUILD}_${RUN_ID}"

mkdir -p "${OUTPUT_PATH}"

echo "Output path is ${OUTPUT_PATH}"
echo "MAX_JOBS=${MAX_JOBS}"

RUN_INFO_FILE="${OUTPUT_BASE}/latest_run.env"
{
    echo "RUN_ID=${RUN_ID}"
    echo "OUTPUT_PATH=${OUTPUT_PATH}"
    echo "PGS=${PGS}"
    echo "BUILD=${BUILD}"
} > "${RUN_INFO_FILE}"


run_chr () {
    chrom=$1

    chr_start=$SECONDS
    echo "Starting chr${chrom}"

    bed_prefix="${INPUT_PATH}/chr${chrom}"
    score_file="${WEIGHTS_PATH}/${PGS}_plink_score_${BUILD}_prepared.txt"
    out_prefix="${OUTPUT_PATH}/${PGS}_score_file_chr${chrom}"

    # per-chromosome log files
    memlog="${OUTPUT_PATH}/mem_chr${chrom}.log"
    plinklog="${OUTPUT_PATH}/plink_chr${chrom}.log"

    # memory monitor (isolated per job)
    (
      while true; do
        date
        grep -E "MemAvailable|MemTotal" /proc/meminfo
        sleep 30
      done
    ) > "${memlog}" &
    MEM_PID=$!

    plink2 \
        --bfile "${bed_prefix}" \
        --score "${score_file}" 1 2 3 \
        --out "${out_prefix}" \
        > "${plinklog}" 2>&1

    kill $MEM_PID || true

    chr_elapsed=$((SECONDS - chr_start))
    printf "chr%s done in %02d:%02d:%02d\n" \
        "${chrom}" \
        $((chr_elapsed / 3600)) \
        $(((chr_elapsed % 3600) / 60)) \
        $((chr_elapsed % 60))
}

export -f run_chr
export INPUT_PATH OUTPUT_PATH WEIGHTS_PATH PGS BUILD PLINK_THREADS

# -----------------------------
# PARALLEL EXECUTION CONTROL
# -----------------------------

for chrom in ${CHROMS}; do
    run_chr "${chrom}" &

    # enforce max parallel jobs
    while [[ $(jobs -r -p | wc -l) -ge ${MAX_JOBS} ]]; do
        sleep 2
    done
done

wait

elapsed=$SECONDS

echo "Finished at $(date)"
printf "All chromosome jobs complete in %02d:%02d:%02d\n" \
    $((elapsed / 3600)) \
    $(((elapsed % 3600) / 60)) \
    $((elapsed % 60))

Overwriting /home/jupyter/workspace/workspace-bucket/calculate_pgs/plink_bed_parallel.sh


# Execute

In [3]:
plink_start = time.time()

max_jobs = max(1, logical_cpus - 2)

env = os.environ.copy()
env.update({
    "PGS": PGS_ID,
    "BUILD": BUILD,
    "CHROMS": CHROMS,
    "MAX_JOBS": str(max_jobs)
})

print("MAX_JOBS:", env["MAX_JOBS"])

script = "/home/jupyter/workspace/workspace-bucket/calculate_pgs/plink_bed_parallel.sh"

subprocess.run(["chmod", "+x", script], check=True)
subprocess.run(["bash", script], check=True, env=env)


print(f"Done; Total time: {(time.time() - plink_start)/60:.1f} minutes")

MAX_JOBS: 62
PLINK JOB START
PGS=PGS002308
BUILD=GRCh38
CHROMS=1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 X
START=Wed Sep  9 01:54:50 PM UTC 2026
Output path is /home/jupyter/workspace/workspace-bucket/calculate_pgs/plink_results/PGS002308_GRCh38_20260909_135450
MAX_JOBS=62
Starting chr1
Starting chr2
Starting chr3
Starting chr4
Starting chr5
Starting chr6
Starting chr7
Starting chr8
Starting chr9
Starting chr10
Starting chr11
Starting chr12
Starting chr13
Starting chr14
Starting chr15
Starting chr16
Starting chr17
Starting chr18
Starting chr19
Starting chr20
Starting chr21
Starting chr22
Starting chrX
chr21 done in 00:42:44
chr22 done in 00:43:42
chr19 done in 01:02:37
chr20 done in 01:08:41
chr18 done in 01:20:28
chr17 done in 01:21:24
chr14 done in 01:31:09
chr16 done in 01:32:48
chr13 done in 01:40:45
chr9 done in 02:05:33
chr10 done in 02:17:33
chr11 done in 02:18:13
chr8 done in 02:35:57
chr7 done in 02:38:15
chr6 done in 02:48:32
chr5 done in 02:49:12
chr4 done in 